In [10]:
# Fetching Lead Quality Report Data 

In [11]:
import pandas as pd
import numpy as np
from gspread_pandas import Spread, conf

In [12]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"


## LEAD VALUE REPORT (BY DATE)

In [ ]:
# Set your target date here
target_date_str = "21/02/2026" 
target_date_dt = pd.to_datetime(target_date_str, dayfirst=True)

target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA', 'AJIN', 'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA', 'ADHIL', 
    'SHABNA', 'FARSANA', 'RINSIYA', 'ARSHAD', 'NAFI'
]



c = conf.get_config(file_name=config_path)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'AGENT'       : 'AGENT',
            'COUNTRY'     : 'REGION',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'PRODUCT 1'   : 'PRODUCT',
            'NAME'        : 'NAME',
            'PHONE NO 1'  : 'PHONE NO',
            'STATUS'      : 'STATUS',
            'DATE'        : 'DATE',
            'QTY'        : 'QTY',
            'VALUE'       : 'VALUE'
        }
        
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            # --- CONDITION 1: Must be on target date ---
            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            subset = subset[subset['DATE'].dt.date == target_date_dt.date()]

            if subset.empty:
                continue

            # subset = subset[
            #     subset['CUSTOMER PATH'].astype(str).str.strip().str.upper() == 'LEAD'
            # ]

            subset = subset[
                ~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])
            ]


            if subset.empty:
                continue

            # Clean remaining string columns
            for col in [ 'PHONE NO', 'CUSTOMER PATH']:
                subset[col] = subset[col].astype(str).str.strip().str.upper()

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")

# --- PROCESSING & PIVOTING ---
if not all_data_list.empty if isinstance(all_data_list, pd.DataFrame) else all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)

    master_df['VALUE'] = pd.to_numeric(master_df['VALUE'], errors='coerce').fillna(0)

    # 2. Total landed leads (Count)
    # total_leads = (
    #     master_df
    #     .groupby('AGENT')['PHONE NO']
    #     .count()
    #     .to_frame('Total Landed Leads')
    # )

    # 3. Total Revenue (Sum of VALUE column)
    total_revenue = (
        master_df
        .groupby('AGENT')['VALUE']
        .sum()
        .to_frame('Total Revenue')
    )

    value_pivot = master_df.pivot_table(
        index='AGENT',
        columns='VALUE',
        values='PHONE NO',
        aggfunc='count',
        fill_value=0
    ).add_suffix('_Count')

    # 5. Combine everything
    final_report = total_leads.join([total_revenue, value_pivot], how='left').fillna(0)

    # 6. Formatting & Totals
    final_report = final_report.sort_index()
    
    # Calculate Grand Total Row
    total_row = final_report.sum()
    final_report.loc['TOTAL'] = total_row

    # --- EXPORT ---
    # Ensure directory exists
    import os
    os.makedirs("./Output_Data", exist_ok=True)
    
    file_path = f"./Output_Data/Lead_Value_Report_{target_date_dt.strftime('%d-%m-%Y')}.xlsx"
    final_report.to_excel(file_path, merge_cells=False)

    print(f"Report for {target_date_dt.date()} generated successfully")
    print(f"Saved to: {file_path}")
    print(f"\nExample Revenue: \n{final_report[['Total Landed Leads', 'Total Revenue']].head()}")

else:
    print(f"No valid records found for {target_date_dt.date()}.")

Report for 2026-02-21 generated successfully
Saved to: ./Output_Data/Lead_Value_Report_21-02-2026.xlsx

Example Revenue: 
         Total Landed Leads  Total Revenue
AGENT                                     
ADHIL                  36.0          910.0
AJIN                   31.0          485.0
AMINA                  11.0          155.0
ARSHAD                 17.0          170.0
BURHANA                44.0          820.0
